# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [5]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

responses = demand(price=14.0)

print(len(responses), responses.sum())

/content/demand_lab
200 161


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



1.

In [13]:
resp_A = demand(30.0)
resp_B = demand(25.0)


In [19]:
n = len(resp_A)
p_hat = resp_A.mean()
print('A: ')
print(f"{n} customers, {resp_A.sum()} bought: p_hat = {p_hat:.3f}, r_hat = 30 * p_hat = {30.0 * p_hat:.2f}")

A: 
239 customers, 101 bought: p_hat = 0.423, r_hat = 30 * p_hat = 12.68


In [20]:
n = len(resp_B)
p_hat = resp_B.mean()
print('B: ')
print(f"{n} customers, {resp_B.sum()} bought: p_hat = {p_hat:.3f}, r_hat = 25 * p_hat = {25.0 * p_hat:.2f}")

B: 
183 customers, 80 bought: p_hat = 0.437, r_hat = 25 * p_hat = 10.93


In [9]:
import numpy as np
import pandas as pd

2.

In [21]:
B = 2000
rng = np.random.default_rng(5030)

def bootstrap_r(responses, price, B=B):
    """B bootstrap values of r_hat."""
    idx = rng.integers(0, len(responses), size=(B, len(responses)))
    return price * responses[idx].mean(axis=1)

def se_formula(responses, price):
    """SE of r_hat from P * sqrt(p_hat (1 - p_hat) / n)."""
    p = responses.mean()
    return price * np.sqrt(p * (1 - p) / len(responses))

reps_A = bootstrap_r(resp_A, 30.0)
reps_B = bootstrap_r(resp_B, 25.0)

pd.DataFrame([
    {"price": 30.0, "r_hat": 30.0 * resp_A.mean(),
     "SE (bootstrap)": reps_A.std(ddof=1), "SE (formula)": se_formula(resp_A, 30.0)},
    {"price": 25.0, "r_hat": 25.0 * resp_B.mean(),
     "SE (bootstrap)": reps_B.std(ddof=1), "SE (formula)": se_formula(resp_B, 25.0)},
]).round(3)

,price,r_hat,SE (bootstrap),SE (formula)
0,30.0,12.678,0.949,0.959
1,25.0,10.929,0.912,0.917


3.

In [22]:
# use in the lab
def ci_table(pairs):
    rows = []
    for P, resp, reps in pairs:
        r_hat, se = P * resp.mean(), se_formula(resp, P)
        lo_b, hi_b = np.percentile(reps, [2.5, 97.5])
        rows.append({"price": P, "n": len(resp), "r_hat": round(r_hat, 2),
                     "95% CI (bootstrap)": f"({lo_b:.2f}, {hi_b:.2f})",
                     "95% CI (r_hat ± 1.96 SE)": f"({r_hat - 1.96 * se:.2f}, {r_hat + 1.96 * se:.2f})"})
    return pd.DataFrame(rows)

ci_table([(30.0, resp_A, reps_A), (25.0, resp_B, reps_B)])

,price,n,r_hat,95% CI (bootstrap),95% CI (r_hat ± 1.96 SE)
0,30.0,239,12.68,"(10.79, 14.56)","(10.80, 14.56)"
1,25.0,183,10.93,"(9.15, 12.70)","(9.13, 12.73)"


4.

In [23]:
price_A = 30.0
price_B = 25.0

def diff_ci(resp_A, price_A, reps_A, resp_B, price_B, reps_B):
    """95% CI for r_A - r_B, from the bootstrap and from the formula."""
    d = price_A * resp_A.mean() - price_B * resp_B.mean()
    boot = tuple(np.percentile(reps_A - reps_B, [2.5, 97.5]))
    se = np.sqrt(se_formula(resp_A, price_A) ** 2 + se_formula(resp_B, price_B) ** 2)
    return d, boot, (d - 1.96 * se, d + 1.96 * se)

def verdict(lo, hi, name_A, name_B):
    if lo > 0:
        return f"{name_A} earns more"
    if hi < 0:
        return f"{name_B} earns more"
    return "can't tell (interval contains 0)"

d, boot, z = diff_ci(resp_A, price_A, reps_A, resp_B, price_B, reps_B)
print(f"r_30 - r_25 = {d:.2f}")
print(f"  95% CI, bootstrap: ({boot[0]:.2f}, {boot[1]:.2f})  ->  {verdict(*boot, '$30', '$25')}")
print(f"  95% CI, formula:   ({z[0]:.2f}, {z[1]:.2f})  ->  {verdict(*z, '$30', '$25')}")

r_30 - r_25 = 1.75
  95% CI, bootstrap: (-0.78, 4.22)  ->  can't tell (interval contains 0)
  95% CI, formula:   (-0.85, 4.35)  ->  can't tell (interval contains 0)


5.

At 30, r_hat = 12.68 and at 25, r_hat = 10.93. SE for A is 0.949 and 0.912 for B. We are 95% confident that r_hat for A is in (10.80, 14.56) and r_hat for B is in (9.13, 12.73). Also, after constructing a confidence interval for the difference, because both intervals contained 0, this sample does not show that 30 earned more revenue than 25. The combination of the r_hats not being many standard errors separate, the confidence intervals overlapping decently, and the bootstrapped difference intervals not showing A making more revenue makes me believe that in this sample, it is difficult to definitively conclude that A is producing more revenue than B.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


7.

In [26]:
# use in the lab
pool_A = np.concatenate([resp_A, demand(price_A)])
pool_B = np.concatenate([resp_B, demand(price_B)])
pool_reps_A = bootstrap_r(pool_A, price_A)
pool_reps_B = bootstrap_r(pool_B, price_B)

print(f"SE ratio (pooled / one test), bootstrap: ${price_A:g}: {pool_reps_A.std(ddof=1) / reps_A.std(ddof=1):.2f}, "
      f"${price_B:g}: {pool_reps_B.std(ddof=1) / reps_B.std(ddof=1):.2f};  1/sqrt(2) = {1 / np.sqrt(2):.2f}")

d, boot, z = diff_ci(pool_A, price_A, pool_reps_A, pool_B, price_B, pool_reps_B)
print(f"pooled r_30 - r_25 = {d:.2f}")
print(f"  95% CI, bootstrap: ({boot[0]:.2f}, {boot[1]:.2f})  ->  {verdict(*boot, '$30', '$25')}")
print(f"  95% CI, formula:   ({z[0]:.2f}, {z[1]:.2f})  ->  {verdict(*z, '$30', '$25')}")

SE ratio (pooled / one test), bootstrap: $30: 0.74, $25: 0.74;  1/sqrt(2) = 0.71
pooled r_30 - r_25 = 1.99
  95% CI, bootstrap: (0.08, 3.94)  ->  $30 earns more
  95% CI, formula:   (0.07, 3.91)  ->  $30 earns more


8.

In [27]:
#had claude help adapt this cell from class code
print(pd.DataFrame([
    {"price": price_A, "n": len(pool_A), "r_hat": price_A * pool_A.mean(),
     "SE (bootstrap)": pool_reps_A.std(ddof=1), "SE (formula)": se_formula(pool_A, price_A),
     "formula SE ratio vs. one test": se_formula(pool_A, price_A) / se_formula(resp_A, price_A)},
    {"price": price_B, "n": len(pool_B), "r_hat": price_B * pool_B.mean(),
     "SE (bootstrap)": pool_reps_B.std(ddof=1), "SE (formula)": se_formula(pool_B, price_B),
     "formula SE ratio vs. one test": se_formula(pool_B, price_B) / se_formula(resp_B, price_B)},
]).round(3))

ci_table([(price_A, pool_A, pool_reps_A), (price_B, pool_B, pool_reps_B)])

   price    n   r_hat  SE (bootstrap)  SE (formula)  \
0   30.0  432  12.569           0.707         0.712   
1   25.0  338  10.577           0.678         0.672   

   formula SE ratio vs. one test  
0                          0.743  
1                          0.733  


,price,n,r_hat,95% CI (bootstrap),95% CI (r_hat ± 1.96 SE)
0,30.0,432,12.57,"(11.25, 13.96)","(11.17, 13.97)"
1,25.0,338,10.58,"(9.25, 11.91)","(9.26, 11.89)"


The SE shrunk by a factor of (0.71/0.95=0.74) and (0.68/0.91=0.74), so ~0.74. The SE scales with 1/sqrtN. Here n is about ~1.8 and 1/sqrt1.8 is ~0.74.

9.

The first round did not settle the question, as the difference interval did include 0. This time, the difference interval did not include 0, which makes me believe that 30 may produce more revenue. However, the difference intervals were just marginally above 0 (0.08, 3.94), (0.07, 3.91). I would say that the evidence now points to 30, but because it is close, I would not definitively conclude it. In a business context I would make this distinction and likely move forward with 30. I do think that the second round of testing was worthwhile, as the first round was inconclusive.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

**Plan**
- I want to attempt a hill climbing algorithm
- steps of 10, smaller if it seems that good values are being approached
-

In [28]:
# use in the lab: run_test(demand, price, data)
data = {}   # price -> list of response arrays, one per market test

def run_test(market, price, data):
    """Run one market test at price, save it, and return all responses at that price so far."""
    data.setdefault(price, []).append(market(price))
    return np.concatenate(data[price])

def tests_run(data):
    """Total number of market tests saved in data."""
    return sum(len(tests) for tests in data.values())

In [29]:
# use in the lab: compare_neighbors(demand, price, h, data)
def compare_neighbors(market, price, h, data):
    """Test price - h and price + h once more each (pooled with earlier tests),
    and return the 95% CI for r(price + h) - r(price - h)."""
    lo_p, hi_p = price - h, price + h
    low, high = run_test(market, lo_p, data), run_test(market, hi_p, data)
    d = hi_p * high.mean() - lo_p * low.mean()
    se = np.sqrt(se_formula(low, lo_p) ** 2 + se_formula(high, hi_p) ** 2)
    ci = (d - 1.96 * se, d + 1.96 * se)
    move = "raise" if ci[0] > 0 else ("lower" if ci[1] < 0 else "can't tell")
    return {"price": price, "compared": f"${lo_p:g} vs ${hi_p:g}", "difference": d,
            "ci_low": ci[0], "ci_high": ci[1], "slope": d / (2 * h), "move": move}


In [32]:
tested = []
def test_one(center_price, step_size=10):
    result = compare_neighbors(demand, center_price, step_size, data)
    tested.append(result)
    print(f"{result['compared']}: difference = {result['difference']:.2f}, "
          f"95% CI ({result['ci_low']:.2f}, {result['ci_high']:.2f}), {result['move']}")

center_price = 30.0
step_size=10.0
same_spot = 0
last_move = None

In [33]:
while tests_run(data) <= 23:
    if center_price - step_size < 5 or center_price + step_size > 100:
        break

    test_one(center_price, step_size)
    move = tested[-1]["move"]
    print(f"center ${center_price:g}, step {step_size:g}, tests used: {tests_run(data)}")

    if move == "can't tell":
        same_spot += 1
        if same_spot == 3:
            if step_size > 5:
                step_size, same_spot = 5, 0
            else:
                print("can't tell even at step 5")
                break
    else:
        if last_move is not None and move != last_move:
            if step_size > 5:
                step_size = 5
            else:
                print("direction flipped, peak found")
                break
        center_price = center_price + step_size if move == "raise" else center_price - step_size
        last_move, same_spot = move, 0

print("final center:", center_price)
print("tests used:", tests_run(data))

$20 vs $40: difference = 7.19, 95% CI (5.07, 9.32), raise
center $30, step 10, tests used: 10
$30 vs $50: difference = 1.50, 95% CI (-0.50, 3.50), can't tell
center $40, step 10, tests used: 12
$30 vs $50: difference = 1.86, 95% CI (0.07, 3.65), raise
center $40, step 10, tests used: 14
$40 vs $60: difference = 0.80, 95% CI (-3.31, 4.91), can't tell
center $50, step 10, tests used: 16
$40 vs $60: difference = 0.48, 95% CI (-2.41, 3.38), can't tell
center $50, step 10, tests used: 18
$40 vs $60: difference = 0.73, 95% CI (-1.69, 3.15), can't tell
center $50, step 10, tests used: 20
$45 vs $55: difference = -0.67, 95% CI (-5.26, 3.92), can't tell
center $50, step 5, tests used: 22
$45 vs $55: difference = -2.37, 95% CI (-5.74, 1.01), can't tell
center $50, step 5, tests used: 24
final center: 50.0
tests used: 24


In [38]:
#this table was constructed with claude assistance
print("price, r_hat, CI low, CI high")
for price in sorted(data):
    if len(data[price]) > 0:
        resp = np.concatenate(data[price])
        r_hat = price * resp.mean()
        se = se_formula(resp, price)
        print("$" + str(price), "|", round(r_hat, 2), "|", round(r_hat - 1.96 * se, 2), "|", round(r_hat + 1.96 * se, 2))

a = np.concatenate(data[50.0])
b = np.concatenate(data[45.0])
diff = 45.0 * b.mean() - 50.0 * a.mean()
se = np.sqrt(se_formula(a, 50.0) ** 2 + se_formula(b, 45.0) ** 2)

print()
print("Difference r(45) - r(50):", round(diff, 2))
print("95% CI for the difference:", round(diff - 1.96 * se, 2), "to", round(diff + 1.96 * se, 2))

price, r_hat, CI low, CI high
$20.0 | 8.18 | 7.26 | 9.11
$30.0 | 12.29 | 11.33 | 13.26
$40.0 | 16.03 | 14.74 | 17.31
$45.0 | 17.76 | 15.29 | 20.24
$50.0 | 14.15 | 12.64 | 15.66
$55.0 | 15.4 | 13.1 | 17.69
$60.0 | 16.76 | 14.71 | 18.8

Difference r(45) - r(50): 3.61
95% CI for the difference: 0.71 to 6.51


  The strategy I chose was was